# Interactive Denoising Lab — GR1

Experiment with GR00T N1.6's flow-matching denoising loop using the bundled
`demo_data/gr1.PickNPlace` dataset. No simulator or server needed.

GR1 uses **joint-angle** actions (not EEF deltas), so the 3D plots show the first
3 joint dimensions as a proxy trajectory. This is useful for comparing denoising
strategies — the shape and spread show how seeds/step counts affect the output —
but the axes represent joint angles, not Cartesian position.

For true EEF trajectory plots, use `interactive_denoising_panda.ipynb` with
`ROBOCASA_PANDA_OMRON` observations captured from the simulator.

In [ ]:
# Cell 1: Imports + config
import sys, os
import numpy as np
import torch
import matplotlib.pyplot as plt

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from scripts.denoising_lab.denoising_lab import (
    DenoisingLab, TrajectoryVisualizer, compare_strategies,
)

MODEL_PATH = "nvidia/GR00T-N1.6-3B"
EMBODIMENT_TAG = "gr1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")

In [ ]:
# Cell 2: Load model
lab = DenoisingLab(MODEL_PATH, EMBODIMENT_TAG, device=DEVICE)
print(f"Model loaded. Action horizon: {lab.action_horizon}, Action dim: {lab.action_dim}")
print(f"Inference timesteps: {lab.num_inference_timesteps}, Timestep buckets: {lab.num_timestep_buckets}")

In [ ]:
# Cell 3: Load observation from bundled demo dataset
import gr00t
from gr00t.data.dataset.lerobot_episode_loader import LeRobotEpisodeLoader
from gr00t.data.dataset.sharded_single_step_dataset import extract_step_data
from gr00t.data.embodiment_tags import EmbodimentTag

DATASET_PATH = os.path.join(
    os.path.dirname(os.path.dirname(gr00t.__file__)),
    "demo_data/gr1.PickNPlace",
)

modality_config = lab.modality_configs
dataset = LeRobotEpisodeLoader(
    dataset_path=DATASET_PATH,
    modality_configs=modality_config,
    video_backend="torchcodec",
    video_backend_kwargs=None,
)

episode_data = dataset[0]
step_data = extract_step_data(
    episode_data, step_index=0,
    modality_configs=modality_config,
    embodiment_tag=EmbodimentTag(EMBODIMENT_TAG),
    allow_padding=False,
)

observation = {
    "video": {k: np.stack(step_data.images[k])[None] for k in step_data.images},
    "state": {k: step_data.states[k][None] for k in step_data.states},
    "language": {
        modality_config["language"].modality_keys[0]: [[step_data.text]],
    },
}

print("Observation loaded.")
print("Video keys:", {k: v.shape for k, v in observation["video"].items()})
print("State keys:", {k: v.shape for k, v in observation["state"].items()})
print("Language:", step_data.text)

In [ ]:
# Cell 4: Encode backbone features (EXPENSIVE — run once)
features = lab.encode_features(observation)
print(f"Backbone features: {features.backbone_features.shape}")
print(f"State features: {features.state_features.shape}")
print(f"Embodiment ID: {features.embodiment_id}")

In [ ]:
# Cell 5: Default denoising (4-step flow matching)
result = lab.denoise(features, seed=42)
print(f"Final action shape: {result.action_pred.shape}")
print(f"Seed: {result.seed}")
for info in result.intermediates:
    print(f"  Step {info.step}: t_cont={info.t_cont:.3f} t_disc={info.t_discretized} "
          f"action_norm={info.action_norm:.4f} velocity_norm={info.velocity_norm:.4f}")

In [ ]:
# Cell 6: Decode + inspect actions
decoded = lab.decode_raw_actions(result.action_pred, features.states)
print("Decoded action keys (GR1 joint groups):")
for key, arr in decoded.items():
    print(f"  {key}: shape={arr.shape}")

for t in range(min(3, list(decoded.values())[0].shape[1])):
    print(lab.label_action_step(decoded, t))

In [ ]:
# Cell 7: 3D joint-space trajectory plot
# GR1 has no EEF key — use the first action key (e.g. "right_arm") as a proxy.
# The 3 axes represent the first 3 joint angles, NOT Cartesian XYZ.
JOINT_KEY = list(decoded.keys())[0]
print(f"Plotting first 3 dims of '{JOINT_KEY}' as a 3D proxy trajectory")

viz = TrajectoryVisualizer()
viz.add_trajectory(decoded, "Default (4-step, seed=42)", eef_key=JOINT_KEY)
fig = viz.plot_eef_3d(title=f"Default Denoising — {JOINT_KEY} (first 3 joints)")
plt.show()

In [ ]:
# Cell 8: Compare seeds — 5 different seeds on the same 3D plot
viz_seeds = TrajectoryVisualizer()
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]

for i, seed in enumerate([0, 42, 123, 456, 789]):
    r = lab.denoise(features, seed=seed)
    d = lab.decode_raw_actions(r.action_pred, features.states)
    viz_seeds.add_trajectory(d, f"seed={seed}", color=colors[i], eef_key=JOINT_KEY)

fig = viz_seeds.plot_eef_3d(title="Seed Comparison — Same Observation, Different Noise")
plt.show()

fig2 = viz_seeds.plot_eef_components(title="Seed Comparison — Joint Components")
plt.show()

In [ ]:
# Cell 9: Compare step counts — 2/4/8/16 denoising steps
strategies = [
    {"num_steps": 2, "seed": 42},
    {"num_steps": 4, "seed": 42},
    {"num_steps": 8, "seed": 42},
    {"num_steps": 16, "seed": 42},
]
labels = ["2 steps", "4 steps", "8 steps", "16 steps"]

results, viz_steps = compare_strategies(lab, features, strategies, labels, eef_key=JOINT_KEY)

fig = viz_steps.plot_eef_3d(title="Step Count Comparison (same seed=42)")
plt.show()

fig2 = viz_steps.plot_eef_components(title="Step Count Comparison — Joint Components")
plt.show()

In [ ]:
# Cell 10: Manual step-by-step denoising
gen = torch.Generator(device=lab.device).manual_seed(42)
actions = torch.randn(
    (1, lab.action_horizon, lab.action_dim),
    dtype=lab.dtype, device=lab.device, generator=gen,
)

num_steps = 4
print(f"Starting manual {num_steps}-step denoising...")
print(f"Initial noise norm: {actions.float().norm().item():.4f}")

for step in range(num_steps):
    velocity, actions = lab.denoise_single_step(features, actions, step, num_steps)
    print(f"  Step {step}: action_norm={actions.float().norm().item():.4f} "
          f"velocity_norm={velocity.float().norm().item():.4f}")

manual_decoded = lab.decode_raw_actions(actions, features.states)
print("\nManual denoising complete. Decoded keys:", list(manual_decoded.keys()))

In [ ]:
# Cell 11: Guided denoising — scale velocity by 1.5x
from functools import partial

def scale_velocity(actions, step_idx, velocity, scale=1.5):
    return velocity * scale

result_guided = lab.denoise(features, seed=42, guided_fn=partial(scale_velocity, scale=1.5))
result_normal = lab.denoise(features, seed=42)

viz_guide = TrajectoryVisualizer()
d_normal = lab.decode_raw_actions(result_normal.action_pred, features.states)
d_guided = lab.decode_raw_actions(result_guided.action_pred, features.states)

viz_guide.add_trajectory(d_normal, "Normal", color="tab:blue", eef_key=JOINT_KEY)
viz_guide.add_trajectory(d_guided, "Guided (1.5x velocity)", color="tab:red", eef_key=JOINT_KEY)

fig = viz_guide.plot_eef_3d(title="Guided vs Normal Denoising")
plt.show()

In [ ]:
# Cell 12: Raw denoising loop — full playground mode
seed = 42
num_steps = 4
dt = 1.0 / num_steps

vl_embeds = features.backbone_features
state_feats = features.state_features
emb_id = features.embodiment_id
bb_output = features.backbone_output
B = vl_embeds.shape[0]
device = vl_embeds.device

gen = torch.Generator(device=device).manual_seed(seed)
actions = torch.randn(
    (B, lab.action_horizon, lab.action_dim),
    dtype=vl_embeds.dtype, device=device, generator=gen,
)

with torch.inference_mode():
    for t in range(num_steps):
        t_cont = t / float(num_steps)
        t_disc = int(t_cont * lab.num_timestep_buckets)

        ts = torch.full((B,), t_disc, device=device)
        act_feat = lab.action_head.action_encoder(actions, ts, emb_id)

        if lab.action_head.config.add_pos_embed:
            pos = torch.arange(act_feat.shape[1], dtype=torch.long, device=device)
            act_feat = act_feat + lab.action_head.position_embedding(pos).unsqueeze(0)

        sa = torch.cat((state_feats, act_feat), dim=1)

        if lab.action_head.config.use_alternate_vl_dit:
            out = lab.action_head.model(
                hidden_states=sa, encoder_hidden_states=vl_embeds, timestep=ts,
                image_mask=bb_output.image_mask,
                backbone_attention_mask=bb_output.backbone_attention_mask,
            )
        else:
            out = lab.action_head.model(
                hidden_states=sa, encoder_hidden_states=vl_embeds, timestep=ts,
            )

        pred = lab.action_head.action_decoder(out, emb_id)
        velocity = pred[:, -lab.action_horizon:]

        # === EDIT HERE: modify velocity, inject guidance, etc. ===
        # velocity = velocity * 1.0  # no-op example

        actions = actions + dt * velocity
        print(f"Step {t}: action_norm={actions.float().norm():.4f} vel_norm={velocity.float().norm():.4f}")

raw_decoded = lab.decode_raw_actions(actions, features.states)
print("\nDone. Decoded keys:", list(raw_decoded.keys()))

In [ ]:
# Cell 13: Denoising progression visualization
result_for_prog = lab.denoise(features, seed=42)
fig = viz.plot_denoising_progression(result_for_prog, lab, eef_key=JOINT_KEY)
plt.show()